# NB-07 — SmartIntake Complete

**Learning goal:** Combine everything from NB-01 to NB-06 into one working system: multi-turn conversation, function calling, Pydantic validation, retry logic, log-safe JSON output, and graceful degradation.

**Concepts covered**
- All five layers working together
- Log-safe output — writing the record without the raw user message
- The `turns_taken` counter
- Graceful degradation — the system never crashes
- Running all five test inputs from the capstone spec

## Cell 1 — Imports and Setup

In [ ]:
from dotenv import load_dotenv
import os
import json
import anthropic
from datetime import datetime
from pathlib import Path
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal
from langchain_anthropic import ChatAnthropic
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = "claude-sonnet-4-6"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
llm = ChatAnthropic(model=MODEL, anthropic_api_key=ANTHROPIC_API_KEY, max_tokens=512)

Path("output").mkdir(exist_ok=True)

## Cell 2 — IntakeRecord Model

In [ ]:
class IntakeRecord(BaseModel):
    query_type: Literal[
        "complaint", "submission", "variation", "safety_signal",
        "label_update", "inspection", "general_enquiry"
    ]
    regulation_ref: Literal[
        "FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
        "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"
    ]
    product_area: Literal[
        "oncology", "cardiovascular", "infectious_disease",
        "cmc", "clinical", "labelling", "general"
    ]
    urgency: Literal["routine", "standard", "urgent", "critical"]
    submitting_team: str

    @field_validator("submitting_team")
    @classmethod
    def team_must_not_be_empty(cls, v):
        if not v.strip():
            raise ValueError("submitting_team cannot be empty")
        return v.strip().title()

    @field_validator("submitting_team")
    @classmethod
    def team_not_a_person(cls, v):
        if len(v.split()) == 1 and v[0].isupper() and v.isalpha():
            raise ValueError("submitting_team must be a team, not an individual name.")
        return v

## Cell 3 — Tool Schema

In [ ]:
INTAKE_TOOL = {
    "name": "extract_intake_fields",
    "description": "Extract structured regulatory intake fields.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query_type": {
                "type": "string",
                "enum": ["complaint", "submission", "variation", "safety_signal",
                         "label_update", "inspection", "general_enquiry"]
            },
            "regulation_ref": {
                "type": "string",
                "enum": ["FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
                         "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"]
            },
            "product_area": {
                "type": "string",
                "enum": ["oncology", "cardiovascular", "infectious_disease",
                         "cmc", "clinical", "labelling", "general"]
            },
            "urgency": {
                "type": "string",
                "enum": ["routine", "standard", "urgent", "critical"]
            },
            "submitting_team": {"type": "string"}
        },
        "required": ["query_type", "regulation_ref", "product_area",
                     "urgency", "submitting_team"]
    }
}

## Cell 4 — System Prompts

In [ ]:
EXTRACTION_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist at PharmaCore India.
Use the extract_intake_fields tool to extract structured fields from the query.

RULES:
- Never infer urgency from tone. Use "routine" if deadline is unclear.
- submitting_team must be a team name, never an individual's name.
- Use "other" for regulation_ref if the framework is not clearly identifiable.
- If the query is not regulatory in nature, do NOT call the tool.
  Instead respond: "I handle pharmaceutical regulatory intake only. [reason]"
- Think step by step about the regulatory context before choosing enum values.

EXAMPLE 1:
User: "We need to respond to an FDA CMC query for NDA-209114."
→ call extract_intake_fields with query_type=submission, regulation_ref=FDA_21CFR,
  product_area=cmc, urgency=routine (no deadline stated), submitting_team=[ask]

EXAMPLE 2:
User: "PV team. Serious unexpected SUSAR for Phase III trial, EMA 15-day deadline, ICH E2A."
→ call extract_intake_fields with query_type=safety_signal, regulation_ref=ICH_E2A,
  product_area=clinical, urgency=critical, submitting_team=PV Team
"""

CONFIRMATION_SYSTEM = """
You are SmartIntake. Generate a brief formal compliance acknowledgement (under 60 words).
Reference the query type and regulation formally.
State the urgency tier.
Name the submitting team.
End with: "Please confirm or correct before this record is saved."
"""

FOLLOWUP_SYSTEM = """
You are SmartIntake. The user has already provided some intake information.
Ask specifically for the missing fields listed. Ask for only one field at a time.
Do not ask for fields already provided. Be concise and professional.
"""

## Cell 5 — Core Functions

In [ ]:
def extract_with_tool(conversation_history: list) -> dict | None:
    """Call Claude with the tool schema. Return extracted fields or None."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=EXTRACTION_SYSTEM,
        tools=[INTAKE_TOOL],
        tool_choice={"type": "auto"},
        messages=conversation_history
    )

    if response.stop_reason == "tool_use":
        for block in response.content:
            if block.type == "tool_use":
                return block.input
    return None


def generate_confirmation(record: IntakeRecord) -> str:
    """Generate a formal confirmation message for the extracted record."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=150,
        system=CONFIRMATION_SYSTEM,
        messages=[{"role": "user", "content": f"Intake record: {record.model_dump_json()}"}]
    )
    return response.content[0].text


def ask_for_missing(missing_fields: list[str], collected_so_far: dict) -> str:
    """Generate a follow-up question for missing fields."""
    prompt = (
        f"Fields already collected: {json.dumps(collected_so_far)}\n"
        f"Fields still needed: {', '.join(missing_fields)}\n"
        "Ask for the first missing field only."
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=150,
        system=FOLLOWUP_SYSTEM,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


def save_record(record: IntakeRecord, turns_taken: int) -> str:
    """Save a log-safe JSON record. Returns the file path."""
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    filepath = f"output/intake_{timestamp}.json"
    data = record.model_dump()
    data["timestamp"] = datetime.now().isoformat()
    data["turns_taken"] = turns_taken
    data["log_safe"] = True           # raw user message is never written here
    with open(filepath, "w") as f:
        json.dump(data, f, indent=2)
    return filepath


def log_debug(message: str):
    """Write to debug.log. Never log raw user messages."""
    with open("debug.log", "a") as f:
        f.write(f"[{datetime.now().isoformat()}] {message}\n")

## Cell 6 — The Session Runner

In [ ]:
def run_session(initial_message: str, additional_turns: list[str] = None):
    """
    Run a SmartIntake session.
    initial_message: the first user message
    additional_turns: optional list of follow-up messages (for partial path testing)
    """
    all_messages = [initial_message] + (additional_turns or [])
    conversation_history = []
    collected = {}
    turns_taken = 0

    print(f"\n{'='*60}")
    print("SmartIntake — Regulatory Intake Session")
    print(f"{'='*60}\n")

    for user_input in all_messages:
        turns_taken += 1
        print(f"[Turn {turns_taken}] User: {user_input}")
        conversation_history.append({"role": "user", "content": user_input})

        # Attempt tool-based extraction
        extracted = extract_with_tool(conversation_history)

        if extracted is None:
            # Claude did not call the tool — non-regulatory or needs clarification
            response = client.messages.create(
                model=MODEL,
                max_tokens=200,
                system=EXTRACTION_SYSTEM,
                messages=conversation_history
            )
            reply = response.content[0].text
            print(f"[Turn {turns_taken}] SmartIntake: {reply}\n")
            conversation_history.append({"role": "assistant", "content": reply})
            log_debug(f"Turn {turns_taken}: no tool call — model replied in text")
            continue

        # Merge with previously collected fields
        collected.update({k: v for k, v in extracted.items() if v})

        # Attempt Pydantic validation
        for attempt in range(2):
            try:
                record = IntakeRecord(**collected)
                # Validation passed
                confirmation = generate_confirmation(record)
                print(f"[Turn {turns_taken}] SmartIntake: {confirmation}\n")
                filepath = save_record(record, turns_taken)
                log_debug(f"Turn {turns_taken}: record saved to {filepath}")
                print(f"✓ Record saved: {filepath}")
                print(json.dumps(record.model_dump(), indent=2))
                return record

            except ValidationError as e:
                missing = [err["loc"][0] for err in e.errors()]
                log_debug(f"Turn {turns_taken} attempt {attempt+1}: missing fields {missing}")

                if attempt == 0:
                    followup = ask_for_missing(missing, collected)
                    print(f"[Turn {turns_taken}] SmartIntake: {followup}\n")
                    conversation_history.append({"role": "assistant", "content": followup})
                else:
                    print(f"[Turn {turns_taken}] SmartIntake: I was unable to extract all "
                          f"required fields automatically. Missing: {missing}. "
                          f"Please provide them directly.\n")
                break

    print("Session ended without a complete record.\n")
    return None

## Cell 7 — Test: Happy Path (T1)

In [ ]:
run_session(
    "PV team here. We have a serious unexpected SUSAR for study NCT-2244 "
    "and need to report to EMA within 15 days per ICH E2A."
)

## Cell 8 — Test: Partial Path Across Three Turns (T2)

In [ ]:
run_session(
    initial_message="Hi, we need to file a Type II variation for our cardiovascular product with the EMA.",
    additional_turns=[
        "The submitting team is Regulatory Affairs Europe.",
        "Urgency is standard — no hard deadline yet."
    ]
)

## Cell 9 — Test: Full Single Turn (T3)

In [ ]:
run_session(
    "CMC Regulatory here. FDA issued a 483 observation on our manufacturing "
    "site in Pune. We have 15 business days to respond."
)

## Cell 10 — Test: Non-Regulatory Fallback (T4)

In [ ]:
run_session("Can you check what the weather is like in Mumbai today?")

## Cell 11 — Test: Partial Urgency (T5)

In [ ]:
run_session(
    initial_message="Labelling team. We need to update the SmPC for our oncology "
                    "product following the latest EMA assessment report.",
    additional_turns=[
        "No specific deadline — standard timeline is fine."
    ]
)

## Cell 12 — Inspect the Output Files

In [ ]:
import os

output_files = sorted(Path("output").glob("intake_*.json"))
print(f"Records saved: {len(output_files)}\n")

for f in output_files:
    print(f"--- {f.name} ---")
    with open(f) as fp:
        print(json.dumps(json.load(fp), indent=2))
    print()

## Cell 13 — Inspect the Debug Log

In [ ]:
print("debug.log contents:")
print("-" * 40)
with open("debug.log") as f:
    print(f.read())

# Verify: raw user messages should NOT appear in the log

### Key Takeaway

NB-07 is the complete SmartIntake system. Every cell from NB-01 to NB-06 lives inside it. The `run_session()` function is the main entry point — it handles all three required paths (happy, partial, fallback), validates with Pydantic, writes log-safe output, and never crashes.